# Sub-Agent Types: Sequential, Parallel & Loop Agents

Companion notebook for the [Sub-Agent Types wiki page](https://ml-viz-ruby.vercel.app/wiki/sub-agent-orchestration).

We build the three deterministic workflow primitives **from scratch** as a tiny combinator
library: every agent is a function `state -> (state, latency, cost)`, and `Sequential`,
`Parallel`, and `Loop` *compose agents into agents*. Then we assemble the research pipeline
from the wiki page and measure how latency and cost differ across the primitives.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

## 1 — Agents as composable functions

A leaf agent simulates one LLM-backed worker: it transforms the state dict and reports a
fixed latency and cost. The three combinators return *new* agents with the aggregation rules
from the wiki page:

- **Sequential** — latency and cost both **add**.
- **Parallel** — cost adds, latency is the **max** of the branches (that's the whole point).
- **Loop** — repeats a body until `done(state)` or `k_max`, accumulating both.

In [ ]:
def leaf(name, latency, cost, update):
    def run(state):
        new = dict(state); new.update(update(state)); new.setdefault('log', [])
        new['log'] = state.get('log', []) + [name]
        return new, latency, cost
    return run

def Sequential(*agents):
    def run(state):
        total_t = total_c = 0.0
        for agent in agents:
            state, t, c = agent(state)
            total_t += t; total_c += c
        return state, total_t, total_c
    return run

def Parallel(*agents):
    def run(state):
        merged, total_t, total_c = dict(state), 0.0, 0.0
        logs = state.get('log', [])
        for agent in agents:                       # branches see the SAME input state
            out, t, c = agent(state)
            for k, v in out.items():               # explicit fan-in merge
                if k != 'log':
                    merged[k] = v
            logs = logs + [e for e in out['log'] if e not in logs]
            total_t = max(total_t, t); total_c += c
        merged['log'] = logs
        return merged, total_t, total_c
    return run

def Loop(body, done, k_max):
    def run(state):
        total_t = total_c = 0.0
        for i in range(k_max):
            state, t, c = body(state)
            total_t += t; total_c += c
            if done(state):
                state = dict(state, converged=True, iterations=i + 1)
                return state, total_t, total_c
        state = dict(state, converged=False, iterations=k_max)   # cap-out: best iterate + flag
        return state, total_t, total_c
    return run

## 2 — Composing the research pipeline

`Sequential(Planner, Parallel(3 researchers), Loop(Writer → Critic), Formatter)` — the
algebra nests because each combinator returns an ordinary agent. The critic approves once
the draft has been revised twice (deterministic stand-in for a quality check).

In [ ]:
planner   = leaf('planner',   2.0, 0.01, lambda s: {'plan': ['web', 'papers', 'data']})
web       = leaf('web',       6.0, 0.02, lambda s: {'web_notes': 'w'})
papers    = leaf('papers',    9.0, 0.03, lambda s: {'paper_notes': 'p'})
data      = leaf('data',      4.0, 0.02, lambda s: {'data_notes': 'd'})
writer    = leaf('writer',    5.0, 0.04, lambda s: {'draft': s.get('draft', 0) + 1})
critic    = leaf('critic',    2.0, 0.01, lambda s: {'approved': s['draft'] >= 2})
formatter = leaf('formatter', 1.0, 0.005, lambda s: {'final': True})

pipeline = Sequential(
    planner,
    Parallel(web, papers, data),
    Loop(Sequential(writer, critic), done=lambda s: s.get('approved'), k_max=3),
    formatter,
)

state, latency, cost = pipeline({'topic': 'agent observability'})
print(f"latency = {latency:.1f} s   cost = ${cost:.3f}")
print(f"loop: converged={state['converged']} after {state['iterations']} iterations")
print('order:', ' → '.join(state['log']))

The gather stage took 9 s (the slowest branch, `papers`) instead of the 19 s a sequential
version would need — while its cost is identical: parallelism buys **latency**, never
**dollars**.

## 3 — The primitives' cost/latency profiles

In [ ]:
gather_seq = Sequential(web, papers, data)
gather_par = Parallel(web, papers, data)

_, t_seq, c_seq = gather_seq({})
_, t_par, c_par = gather_par({})

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, vals, title, unit in [
    (axes[0], (t_seq, t_par), 'Latency of the gather stage', 's'),
    (axes[1], (c_seq, c_par), 'Cost of the gather stage', '$'),
]:
    bars = ax.bar(['Sequential', 'Parallel'], vals, color=['#f59e0b', '#2dd4bf'], width=0.55)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v, f'{unit}{v:.2f}' if unit == '$' else f'{v:.0f}{unit}',
                ha='center', va='bottom', fontsize=10)
    ax.set_title(title, fontsize=11)
plt.suptitle('Parallel fan-out: latency = max of branches, cost still = sum', y=1.02)
plt.tight_layout(); plt.show()

## ✏️ Your turn

**Concept recap.** A loop agent's exit condition can *never fire* on some input — that's why
the cap lives in the runtime. But a plain iteration cap wastes money when the body stops
making progress long before `k_max`: if an iteration leaves the state unchanged, iterating
again is pointless.

**Exercise.** Implement `LoopUntilStable(body, k_max)`: run `body` repeatedly, but stop early
when an iteration produces **no change in state** (compare the dicts, ignoring the `log`
key). Return the state with `iterations` set, like `Loop`.

In [ ]:
def strip_log(state):
    return {k: v for k, v in state.items() if k != 'log'}

def LoopUntilStable(body, k_max):
    def run(state):
        total_t = total_c = 0.0
        for i in range(k_max):
            new_state, t, c = body(state)
            total_t += t; total_c += c
            # TODO(you): if strip_log(new_state) == strip_log(state), stop and
            #            return with iterations = i + 1; otherwise continue
            state = new_state
        return dict(state, iterations=k_max), total_t, total_c
    return run

In [ ]:
# This assert cell passes silently when your implementation is correct.
capped = leaf('inc', 1.0, 0.01, lambda s: {'x': min(s.get('x', 0) + 1, 3)})   # stabilises at x=3
out, t, c = LoopUntilStable(capped, k_max=10)({'x': 0})
assert out['x'] == 3
assert out['iterations'] == 4, f"expected 4 iterations (3 changes + 1 stable), got {out['iterations']}"
assert abs(t - 4.0) < 1e-9 and abs(c - 0.04) < 1e-9

never_stable = leaf('grow', 1.0, 0.01, lambda s: {'x': s.get('x', 0) + 1})
out2, t2, _ = LoopUntilStable(never_stable, k_max=5)({'x': 0})
assert out2['iterations'] == 5 and out2['x'] == 5      # cap still guarantees termination
print('✓ LoopUntilStable exits early on a fixed point and still respects k_max')

<details>
<summary>Solution</summary>

```python
def LoopUntilStable(body, k_max):
    def run(state):
        total_t = total_c = 0.0
        for i in range(k_max):
            new_state, t, c = body(state)
            total_t += t; total_c += c
            if strip_log(new_state) == strip_log(state):
                return dict(new_state, iterations=i + 1), total_t, total_c
            state = new_state
        return dict(state, iterations=k_max), total_t, total_c
    return run
```

Two guards compose: the fixed-point check saves money when the body converges, and `k_max`
remains the hard guarantee for bodies that never do. That's the wiki page's rule in code —
the prompt (or the convergence heuristic) is a suggestion; the runtime cap is a guarantee.
</details>